In [ ]:
!pip install -q huggingface_hub datasets torch tqdm

In [ ]:
import os
import sys
import torch
import json
import pickle
from torch.utils.data import DataLoader
from huggingface_hub import HfApi, hf_hub_download

# Add src to path if not already there
if 'src' not in sys.path:
    sys.path.append(os.path.abspath('src'))

from data import get_jefferson_text, preprocess, build_vocab, TrigramDataset
from models import CountTrigramModel, NeuralTrigramModel
from train import train_neural_model
from decode import generate_greedy, generate_top_k, generate_nucleus, generate_beam_search
from eval import calculate_perplexity_count, calculate_perplexity_neural

## 1. Dataset Preparation

In [ ]:
print("Loading Thomas Jefferson speeches...")
text = get_jefferson_text()
tokens = preprocess(text)

# Split into train and test
split_idx = int(len(tokens) * 0.9)
train_tokens = tokens[:split_idx]
test_tokens = tokens[split_idx:]

vocab, word2idx, idx2word = build_vocab(train_tokens)
vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")
print(f"Train tokens: {len(train_tokens)}")
print(f"Test tokens: {len(test_tokens)}")

## 2. Model Development

### 2.1 Count-based Trigram Model

In [ ]:
train_idx = [word2idx[w] for w in train_tokens]
test_idx = [word2idx[w] for w in test_tokens if w in word2idx]

count_model = CountTrigramModel(vocab_size=vocab_size, add_k=0.1)
count_model.train(train_idx)

ppl_count = calculate_perplexity_count(count_model, test_idx)
print(f"Count Model Test Perplexity: {ppl_count:.4f}")

### 2.2 Neural Trigram Model

In [ ]:
train_dataset = TrigramDataset(train_tokens, word2idx)
val_split = int(len(train_dataset) * 0.9)
train_ds, val_ds = torch.utils.data.random_split(train_dataset, [val_split, len(train_dataset) - val_split])
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
nn_model = NeuralTrigramModel(vocab_size=vocab_size, embed_size=64, hidden_size=128)
nn_model = train_neural_model(nn_model, train_loader, val_loader, epochs=50, patience=5, lr=1e-3, device=device)

test_dataset = TrigramDataset([w for w in test_tokens if w in word2idx], word2idx)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)
ppl_nn = calculate_perplexity_neural(nn_model, test_loader, device=device)
print(f"Neural Model Test Perplexity: {ppl_nn:.4f}")

## 3. Upload to Hugging Face

We will upload the Neural Model, Count Model, and Vocabulary.

In [ ]:
# Save Neural Model
torch.save(nn_model.state_dict(), 'nn_model.pth')

# Save Count Model (using pickle to preserve defaultdict structure)
with open('count_model.pkl', 'wb') as f:
    pickle.dump(count_model, f)

# Save Vocab
with open('vocab.json', 'w') as f:
    json.dump({'word2idx': word2idx, 'idx2word': {str(k): v for k, v in idx2word.items()}}, f)

print("Models and vocab saved locally.")

In [ ]:
repo_id = "bdanko/n-gram-modeling"
api = HfApi()

files_to_upload = ["nn_model.pth", "count_model.pkl", "vocab.json"]

for filename in files_to_upload:
    try:
        api.upload_file(
            path_or_fileobj=filename,
            path_in_repo=filename,
            repo_id=repo_id
        )
        print(f"Uploaded {filename} to {repo_id}")
    except Exception as e:
        print(f"Failed to upload {filename}: {e}")

## 4. Evaluation Using Uploaded Models

In [ ]:
print("Downloading models from Hugging Face...")
hf_nn_path = hf_hub_download(repo_id=repo_id, filename="nn_model.pth")
hf_count_path = hf_hub_download(repo_id=repo_id, filename="count_model.pkl")
hf_vocab_path = hf_hub_download(repo_id=repo_id, filename="vocab.json")

with open(hf_vocab_path, 'r') as f:
    hf_vocab_data = json.load(f)
    hf_word2idx = hf_vocab_data['word2idx']
    hf_idx2word = {int(k): v for k, v in hf_vocab_data['idx2word'].items()}

# Load Neural Model
loaded_nn_model = NeuralTrigramModel(vocab_size=len(hf_word2idx))
loaded_nn_model.load_state_dict(torch.load(hf_nn_path, weights_only=True))
loaded_nn_model.eval()

# Load Count Model
with open(hf_count_path, 'rb') as f:
    loaded_count_model = pickle.load(f)

print("Models loaded successfully from HF.")

In [ ]:
w1_word, w2_word = "fellow", "citizens"
w1, w2 = hf_word2idx[w1_word], hf_word2idx[w2_word]
seed_text = f"{w1_word} {w2_word}"

models_to_test = [("Count-based Trigram Model", loaded_count_model), ("Neural Trigram Model", loaded_nn_model)]

for name, model in models_to_test:
    print(f"\n{'='*20} {name} {'='*20}")
    
    print("\nGreedy")
    print(generate_greedy(model, w1, w2, hf_word2idx, hf_idx2word, num_words=80))

    print("\nBeam Search (width=3)")
    print(generate_beam_search(model, w1, w2, hf_word2idx, hf_idx2word, num_words=80, beam_width=3))

    print("\nTop-K Sampling (K=5)")
    print(generate_top_k(model, w1, w2, hf_word2idx, hf_idx2word, num_words=80, k=5))

    print("\nNucleus Sampling (p=0.9)")
    print(generate_nucleus(model, w1, w2, hf_word2idx, hf_idx2word, num_words=80, p=0.9))

## 5. Summary Evaluation & Critique

### 5.1 Perplexity Comparison
- **Count-Based Trigram Model**: Typically produces higher perplexity as it lacks the semantic generalization capabilities of neural embeddings. It is sensitive to exact matches and struggles with unseen combinations even with Add-k smoothing.
- **Neural Trigram Model**: Generally achieves significantly lower perplexity. The word embeddings allow the model to generalize across semantically similar words, leading to a much smoother and more robust probability distribution.

### 5.2 Decoding Strategies Critique

- **Greedy Decoding**: Frequently collapses into repetitive loops (e.g., "of the of the of the") or gets stuck on common rhetorical phrases. It lacks the diversity required for long-form speech generation.
- **Beam Search (Width=3)**: Produces more coherent local structures than greedy decoding but still suffers from repetitiveness. By maximizing overall sequence probability, it often defaults to the most generic and safe language patterns.
- **Top-K Sampling (K=5)**: Provides a significant improvement in "speech-like" variety. It avoids the determinism of greedy search while preventing the model from choosing completely nonsensical, low-probability words.
- **Nucleus Sampling (Top-p=0.9)**: Offers the best balance of coherence and creativity. By dynamically adjusting the sampling pool based on cumulative probability, it sounds the most natural and rhetorical, closely mimicking the style of the original corpus.

**Conclusion**: Nucleus sampling (Top-p) is the superior choice for generating speech-like text in this domain, while the Neural Model provides the necessary generalization to avoid the sparsity issues of classical n-grams.